# Generate report for tax purposes

The template for celiac tax credit is usually stuctured like this: 
item type, quantity, non-gf average cost, average gf-cost, average incremental cost, total claim

Mapping: 
- item type: reference item name
- quantity: count expenses group by reference item
- non-gf average cost: reference item price
- average gf cost: ... see below
- average incremental cost: difference of non-gf and gf
- total: count * difference

Note on average gf cost: In my case I don't count instances of expenses (e.g. 1x hot dog buns) but I adjust it to quantity/weight (e.g. 4 hot dog buns). To compare apples to apples, I need the expense amount adjusted to the weight/count of the reference item. That means that for each expense I need to calculate the price the item would be if it was the weight of the reference item but with the price per weight of the expense using the weight of product associated with the expense. 

In [1]:
%pip install -q \
    psycopg2-binary \
    pandas \
    openpyxl


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2
import pandas as pd

In [3]:
# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="receipts_app",
    user="postgres",
    password="postgres",
    options="-c search_path=receipts_app"
)

# Pre-flight

In [4]:
cur = conn.cursor()


## Check for missing price proofs

In [6]:
with open('../nifi_product_price_proof_request.sql', 'r') as f:
    query = f.read()
cur.execute(query)

# Fetch results
results = cur.fetchall()

print(f"Found {len(results)} reference items needing price proofs:")
for row in results:
    print(f"Reference Item ID: {row[0]}")

Found 0 reference items needing price proofs:


## Check for missing conversions

In [13]:
with open('../current_conversions_needed.sql', 'r') as f:
    query = f.read()
cur.execute(query)

# Fetch results
results = cur.fetchall()

print(f"Found {len(results)} Reference items with missing conversions (product vs proof):")
for row in results:
    print(f"Reference items: {row}")

Found 0 Reference items with missing conversions (product vs proof):


# Report

## General clean-up notes

- I noticed a few $0 expenses, manually investigated and fixed
- Picking up mistakes like a $999 item (decimal point not dected in OCR)
- Conversion on gum was incorrect (fixed it by setting it the same in the product)

## CRA-style report

In [14]:
with open('../report.sql', 'r') as f:
    query = f.read()
cur.execute(query)

# Fetch results
results = cur.fetchall()

df = pd.DataFrame(results, columns=[
    'reference_item_id', 
	'reference_item_per_id',
	'quantity',
	'non_gf_average_cost',
	'gf_average_cost',
	'non_gf_average_cost',
    'avg_incr_cost',
    'total_cost'
])

df = df.drop(columns=['reference_item_id'])
cost_columns = list(set(col for col in df.columns if col.endswith('_cost')))
for col in cost_columns:
    df[col] = df[col].astype(float).map('${:,.2f}'.format)

df

,reference_item_per_id,quantity,non_gf_average_cost,gf_average_cost,non_gf_average_cost,avg_incr_cost,total_cost
0,BBQ Sauce,1,$1.69,$4.49,$1.69,$2.80,$2.80
1,Beans,8,$1.38,$1.92,$1.38,$0.54,$4.28
2,Bouillon cubes,8.0,$1.03,$4.57,$1.03,$3.54,$28.28
3,Candy,27.0,$2.47,$6.17,$2.47,$3.70,$99.98
4,Cereal,42.0,$5.36,$10.57,$5.36,$5.21,$218.97
5,Chia seeds,4.0,$5.24,$9.19,$5.24,$3.95,$15.81
6,Chips,44.0,$2.41,$5.40,$2.41,$2.99,$131.52
7,Chocolate,23.0,$0.93,$4.41,$0.93,$3.48,$80.07
8,Chocolate chips,15.0,$2.69,$7.48,$2.69,$4.79,$71.90
9,Cookies,18.0,$1.01,$7.24,$1.01,$6.23,$112.17


In [15]:
total_cost_sum = df['total_cost'].replace({'\$': '', ',': ''}, regex=True).astype(float).sum()
print(f'Total of total_cost column: ${total_cost_sum:,.2f}')


Total of total_cost column: $5,843.45


<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_31164/1169038756.py:1: SyntaxWarning: invalid escape sequence '\$'
  total_cost_sum = df['total_cost'].replace({'\$': '', ',': ''}, regex=True).astype(float).sum()


In [16]:
# Export the DataFrame to an Excel file
df.to_excel('../expense_report.xlsx', index=False)


## LLM verify

In [148]:
prompt = f"""You are an experienced tax auditor specializing in expense deductions, particularly for celiac disease tax credits in Canada. Your task is to review the following expense report and assess each item's eligibility for the celiac disease tax credit.

For each listed item:
1. Verify if the gluten-free product and its regular counterpart are properly categorized
2. Analyze if the price differential between the gluten-free version and regular version is reasonable based on market standards
3. Flag any items where the price difference appears suspicious (either too large or too small)
4. Confirm that only the price difference between gluten-free and regular products is being claimed, not the full cost
5. Check if the items qualify as "medical food" under tax regulations (essential for basic nutrition, not just specialty or luxury items)

Please provide your analysis in a structured format:
- APPROVED: Items that appear legitimate and properly documented
- REQUIRES CLARIFICATION: Items where additional information is needed
- REJECTED: Items that clearly do not qualify for the tax credit

For any flagged items, explain specifically what concerns you have and what additional documentation would be required to approve the claim.

Here is the expense report to review:

{df[['reference_item_per_id', 'quantity', 'non_gf_average_cost', 'gf_average_cost']].to_csv(index=False)}
"""
print(prompt)




You are an experienced tax auditor specializing in expense deductions, particularly for celiac disease tax credits in Canada. Your task is to review the following expense report and assess each item's eligibility for the celiac disease tax credit.

For each listed item:
1. Verify if the gluten-free product and its regular counterpart are properly categorized
2. Analyze if the price differential between the gluten-free version and regular version is reasonable based on market standards
3. Flag any items where the price difference appears suspicious (either too large or too small)
4. Confirm that only the price difference between gluten-free and regular products is being claimed, not the full cost
5. Check if the items qualify as "medical food" under tax regulations (essential for basic nutrition, not just specialty or luxury items)

Please provide your analysis in a structured format:
- APPROVED: Items that appear legitimate and properly documented
- REQUIRES CLARIFICATION: Items where 

# Cleanup

In [17]:
# Close connection
cur.close()
conn.close()